In [7]:
import os, re, plistlib, cv2
import numpy as np
import pydicom
from pathlib import Path
from tqdm import tqdm

# === 路径设置 ===
dcm_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\INbreast\INbreast Release 1.0\AllDICOMs")
xml_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\INbreast\INbreast Release 1.0\AllXML")

out_img_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\images")
out_mask_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\labels")
out_img_dir.mkdir(parents=True, exist_ok=True)
out_mask_dir.mkdir(parents=True, exist_ok=True)

# === 工具函数 ===
def parse_point_px(str_list):
    """解析 ['(x, y)', '(x, y)', ...] 为 (N,2) 数组"""
    pts = []
    for s in str_list:
        nums = [float(t) for t in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)]
        if len(nums) >= 2:
            pts.append([nums[0], nums[1]])
    return np.array(pts, dtype=float)

def xml_to_mask(xml_path, img_shape):
    """从 INbreast XML 文件生成二值 mask"""
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    try:
        with open(xml_path, "rb") as f:
            pl = plistlib.load(f)
    except Exception as e:
        print(f"[跳过] 无法解析 {xml_path.name}: {e}")
        return mask

    for img in pl.get("Images", []):
        for r in img.get("ROIs", []):
            pts = parse_point_px(r.get("Point_px", []))
            if len(pts) > 2:
                pts = np.round(pts).astype(np.int32)
                cv2.fillPoly(mask, [pts], 255)
    return mask


# === 建立 XML 文件索引（按前8位数字）===
xml_map = {x.stem[:8]: x for x in xml_dir.glob("*.xml")}
print(f"找到 XML 文件 {len(xml_map)} 个\n")

# === 遍历 DICOM，按前8位匹配 XML ===
dcm_files = list(dcm_dir.glob("*.dcm"))
print(f"找到 DICOM 文件 {len(dcm_files)} 个\n")

for dcm_path in tqdm(dcm_files, desc="Processing DICOMs"):
    base = dcm_path.stem
    prefix = base[:8]  # 前8位匹配
    xml_path = xml_map.get(prefix)

    # === 读取 DICOM ===
    try:
        dcm = pydicom.dcmread(dcm_path)
        img = dcm.pixel_array.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = (img * 255).astype(np.uint8)
    except Exception as e:
        print(f"[跳过] 无法读取 {dcm_path.name}: {e}")
        continue

    # === 保存图像（JPG） ===
    jpg_path = out_img_dir / f"{base}.jpg"
    cv2.imwrite(str(jpg_path), img)

    # === 匹配到 XML 就生成 mask ===
    if xml_path and xml_path.exists():
        mask = xml_to_mask(xml_path, img.shape)
        mask_path = out_mask_dir / f"{base}_mask.png"
        cv2.imwrite(str(mask_path), mask)
    else:
        print(f"[警告] 找不到对应 XML: {prefix}.xml")

print("\n✅ 全部转换完成！")
print(f"图像输出路径: {out_img_dir}")
print(f"mask 输出路径: {out_mask_dir}")




找到 XML 文件 343 个

找到 DICOM 文件 410 个



Processing DICOMs:   9%|▊         | 35/410 [00:01<00:18, 20.74it/s]

[警告] 找不到对应 XML: 20588138.xml
[警告] 找不到对应 XML: 20588164.xml


Processing DICOMs:  16%|█▌        | 66/410 [00:03<00:16, 21.24it/s]

[警告] 找不到对应 XML: 22580218.xml
[警告] 找不到对应 XML: 22580270.xml


...

[警告] 找不到对应 XML: 53582540.xml
[警告] 找不到对应 XML: 53582567.xml


Processing DICOMs:  95%|█████████▍| 388/410 [00:21<00:00, 22.86it/s]

[警告] 找不到对应 XML: 53586361.xml
[警告] 找不到对应 XML: 53586388.xml
[警告] 找不到对应 XML: 53586415.xml
[警告] 找不到对应 XML: 53586442.xml


Processing DICOMs: 100%|██████████| 410/410 [00:23<00:00, 17.49it/s]

[警告] 找不到对应 XML: 53587690.xml
[警告] 找不到对应 XML: 53587744.xml

✅ 全部转换完成！
图像输出路径: D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\images
mask 输出路径: D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\labels


In [8]:
from pathlib import Path
import os

# === 路径设置 ===
img_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\images")
label_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\labels")

# === 获取 label 的前8位编号集合 ===
label_prefixes = {p.stem[:8] for p in label_dir.glob("*.*")}

deleted = []

# === 遍历 image，删除没有对应 label 的文件 ===
for img in img_dir.glob("*.*"):
    prefix = img.stem[:8]
    if prefix not in label_prefixes:
        try:
            os.remove(img)
            deleted.append(img.name)
        except Exception as e:
            print(f"⚠️ 无法删除 {img.name}: {e}")

print(f"\n✅ 已删除 {len(deleted)} 个 image 文件（无对应 label）")
if deleted:
    print("示例：", ", ".join(deleted[:10]), "..." if len(deleted) > 10 else "")



✅ 已删除 67 个 image 文件（无对应 label）
示例： 20588138_8d0b9620c53c0268_MG_R_ML_ANON.jpg, 20588164_8d0b9620c53c0268_MG_R_CC_ANON.jpg, 22580218_5530d5782fc89dd7_MG_L_CC_ANON.jpg, 22580270_5530d5782fc89dd7_MG_L_ML_ANON.jpg, 22613848_45c7f44839fd9e68_MG_L_ML_ANON.jpg, 22613944_f23fa352e7de3dc7_MG_L_CC_ANON.jpg, 22613996_f23fa352e7de3dc7_MG_L_ML_ANON.jpg, 22670442_7e677f3d530e41ed_MG_R_CC_ANON.jpg, 22670488_7e677f3d530e41ed_MG_R_ML_ANON.jpg, 22670643_e15a16f87b4f9782_MG_L_CC_ANON.jpg ...


In [9]:
import shutil
import random
from pathlib import Path

# === 原始数据路径 ===
orig_img_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\images")
orig_label_dir = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_original\labels")

# === 目标路径（Unet 训练集） ===
target_root = Path(r"D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_Unet")

# 创建目标文件夹结构
splits = ["train", "test", "val"]
for split in splits:
    (target_root / split / "images").mkdir(parents=True, exist_ok=True)
    (target_root / split / "labels").mkdir(parents=True, exist_ok=True)

# === 获取图像列表（按前8位匹配） ===
imgs = sorted(list(orig_img_dir.glob("*.*")))
random.shuffle(imgs)

# === 按比例划分 ===
n_total = len(imgs)
n_train = int(0.8 * n_total)
n_test = int(0.1 * n_total)
n_val = n_total - n_train - n_test

train_imgs = imgs[:n_train]
test_imgs = imgs[n_train:n_train + n_test]
val_imgs = imgs[n_train + n_test:]

print(f"总图像数: {n_total}")
print(f"训练集: {len(train_imgs)}, 测试集: {len(test_imgs)}, 验证集: {len(val_imgs)}")

def copy_pair(img_list, subset_name):
    """复制 image 和对应 label"""
    for img_path in img_list:
        prefix = img_path.stem[:8]
        label_path = orig_label_dir / f"{img_path.stem}_mask.png"

        # 若不存在带后缀_mask的，则尝试直接匹配前8位
        if not label_path.exists():
            matches = list(orig_label_dir.glob(f"{prefix}*.png"))
            if matches:
                label_path = matches[0]
            else:
                print(f"[⚠️ 缺少label] {img_path.name}")
                continue

        # 拷贝 image & label
        shutil.copy(img_path, target_root / subset_name / "images" / img_path.name)
        shutil.copy(label_path, target_root / subset_name / "labels" / label_path.name)

# === 执行划分 ===
copy_pair(train_imgs, "train")
copy_pair(test_imgs, "test")
copy_pair(val_imgs, "val")

print("\n✅ 数据划分完成！")
print(f"train/test/val 数据集已生成在: {target_root}")


总图像数: 343
训练集: 274, 测试集: 34, 验证集: 35

✅ 数据划分完成！
train/test/val 数据集已生成在: D:\OneDrive\OneDrive - Rose-Hulman Institute of Technology\Rose-Hulman\course\CSSE\CSSE416\dataset_Unet
